# SE(3) Equivariance for Protein Generation

This notebook explores the mathematical foundations of SE(3) equivariance and demonstrates how to implement SE(3)-equivariant operations for protein structure generation.

## Learning Objectives

1. Understand the SE(3) group (rotations + translations)
2. Implement rotation representations (quaternions, axis-angle)
3. Build SE(3)-invariant and equivariant features
4. Visualize diffusion on SO(3)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. The SE(3) Group

SE(3) is the **Special Euclidean Group** in 3D, consisting of:
- **SO(3)**: All 3D rotations (orthogonal matrices with determinant 1)
- **$\mathbb{R}^3$**: All 3D translations

A transformation $T = (R, \vec{t})$ acts on a point $\vec{x}$ as:
$$T \cdot \vec{x} = R\vec{x} + \vec{t}$$

In [ ]:
class RigidTransform:
    """Rigid body transformation (rotation + translation)."""
    
    def __init__(self, rotation, translation):
        """
        Args:
            rotation: [..., 3, 3] rotation matrix
            translation: [..., 3] translation vector
        """
        self.rot = rotation
        self.trans = translation
        
    @classmethod
    def identity(cls, batch_shape=()):
        """Create identity transformation."""
        rot = torch.eye(3).expand(*batch_shape, 3, 3).clone()
        trans = torch.zeros(*batch_shape, 3)
        return cls(rot, trans)
    
    def apply(self, points):
        """Apply transformation to points: R @ x + t"""
        return torch.einsum('...ij,...j->...i', self.rot, points) + self.trans
    
    def compose(self, other):
        """Compose transformations: self * other"""
        new_rot = torch.einsum('...ij,...jk->...ik', self.rot, other.rot)
        new_trans = torch.einsum('...ij,...j->...i', self.rot, other.trans) + self.trans
        return RigidTransform(new_rot, new_trans)
    
    def inverse(self):
        """Compute inverse transformation."""
        inv_rot = self.rot.transpose(-1, -2)
        inv_trans = -torch.einsum('...ij,...j->...i', inv_rot, self.trans)
        return RigidTransform(inv_rot, inv_trans)
    
    def __repr__(self):
        return f"RigidTransform(rot shape={self.rot.shape}, trans shape={self.trans.shape})"

# Test identity
T = RigidTransform.identity()
print(T)
print(f"Identity rotation:\n{T.rot}")
print(f"Identity translation: {T.trans}")

In [ ]:
# Test transformation operations
# Create a rotation around z-axis by 90 degrees
angle = np.pi / 2
rot_z = torch.tensor([
    [np.cos(angle), -np.sin(angle), 0],
    [np.sin(angle), np.cos(angle), 0],
    [0, 0, 1]
], dtype=torch.float32)

T1 = RigidTransform(rot_z, torch.tensor([1.0, 0.0, 0.0]))

# Test point transformation
point = torch.tensor([1.0, 0.0, 0.0])
transformed = T1.apply(point)

print(f"Original point: {point}")
print(f"After rotation by 90 deg around z + translation: {transformed}")
# Expected: [1, 1, 0] (rotated to [0, 1, 0] then translated by [1, 0, 0])

# Test composition
T2 = RigidTransform(rot_z, torch.tensor([0.0, 1.0, 0.0]))
T_composed = T1.compose(T2)

# Test inverse
T_inv = T1.inverse()
T_identity = T1.compose(T_inv)

print(f"\nT * T^(-1) rotation (should be identity):\n{T_identity.rot}")

## 2. Rotation Representations

There are multiple ways to represent rotations:

| Representation | Dimensions | Pros | Cons |
|----------------|------------|------|------|
| Rotation Matrix | 9 (3x3) | Easy to apply | Over-parameterized |
| Quaternion | 4 | Compact, no gimbal lock | Unit constraint |
| Axis-Angle | 3 | Minimal, intuitive | Singularity at 0 |
| Euler Angles | 3 | Human-interpretable | Gimbal lock |

In [ ]:
def quaternion_to_rotation_matrix(q):
    """
    Convert quaternion to rotation matrix.
    
    Args:
        q: [..., 4] quaternion (w, x, y, z)
    
    Returns:
        R: [..., 3, 3] rotation matrix
    """
    # Normalize
    q = q / (q.norm(dim=-1, keepdim=True) + 1e-8)
    
    w, x, y, z = q.unbind(-1)
    
    R = torch.stack([
        torch.stack([1 - 2*y*y - 2*z*z, 2*x*y - 2*w*z, 2*x*z + 2*w*y], dim=-1),
        torch.stack([2*x*y + 2*w*z, 1 - 2*x*x - 2*z*z, 2*y*z - 2*w*x], dim=-1),
        torch.stack([2*x*z - 2*w*y, 2*y*z + 2*w*x, 1 - 2*x*x - 2*y*y], dim=-1)
    ], dim=-2)
    
    return R

def axis_angle_to_rotation_matrix(axis_angle):
    """
    Convert axis-angle to rotation matrix using Rodrigues formula.
    
    Args:
        axis_angle: [..., 3] axis * angle
    
    Returns:
        R: [..., 3, 3] rotation matrix
    """
    angle = axis_angle.norm(dim=-1, keepdim=True)
    axis = axis_angle / (angle + 1e-8)
    
    # Skew-symmetric matrix
    K = torch.zeros(*axis_angle.shape[:-1], 3, 3, device=axis_angle.device)
    K[..., 0, 1] = -axis[..., 2]
    K[..., 0, 2] = axis[..., 1]
    K[..., 1, 0] = axis[..., 2]
    K[..., 1, 2] = -axis[..., 0]
    K[..., 2, 0] = -axis[..., 1]
    K[..., 2, 1] = axis[..., 0]
    
    # Identity
    I = torch.eye(3, device=axis_angle.device).expand(*axis_angle.shape[:-1], 3, 3)
    
    # Rodrigues formula: R = I + sin(theta) * K + (1 - cos(theta)) * K^2
    sin_angle = torch.sin(angle).unsqueeze(-1)
    cos_angle = torch.cos(angle).unsqueeze(-1)
    
    R = I + sin_angle * K + (1 - cos_angle) * (K @ K)
    
    return R

# Test conversions
# Quaternion for 90 degree rotation around z-axis
q = torch.tensor([np.cos(np.pi/4), 0, 0, np.sin(np.pi/4)])  # w, x, y, z
R_from_q = quaternion_to_rotation_matrix(q)

# Axis-angle for same rotation
aa = torch.tensor([0, 0, np.pi/2])  # axis * angle
R_from_aa = axis_angle_to_rotation_matrix(aa)

print("Rotation from quaternion:")
print(R_from_q.numpy().round(4))
print("\nRotation from axis-angle:")
print(R_from_aa.numpy().round(4))

## 3. Visualizing SO(3)

Let's visualize how rotations transform a reference frame.

In [ ]:
def plot_frame(ax, origin, rotation, scale=1.0, alpha=1.0, label=None):
    """Plot a coordinate frame (3 arrows for x, y, z axes)."""
    colors = ['red', 'green', 'blue']
    labels = ['X', 'Y', 'Z'] if label is None else [f"{label}-X", f"{label}-Y", f"{label}-Z"]
    
    for i, (color, lbl) in enumerate(zip(colors, labels)):
        direction = rotation[:, i] * scale
        ax.quiver(*origin, *direction, color=color, alpha=alpha, arrow_length_ratio=0.1)

def visualize_rotations():
    """Visualize different rotations."""
    fig = plt.figure(figsize=(15, 5))
    
    # Identity
    ax1 = fig.add_subplot(131, projection='3d')
    ax1.set_title('Identity Rotation')
    plot_frame(ax1, [0, 0, 0], np.eye(3))
    ax1.set_xlim([-1, 1]); ax1.set_ylim([-1, 1]); ax1.set_zlim([-1, 1])
    
    # 45 degree rotation around z
    ax2 = fig.add_subplot(132, projection='3d')
    ax2.set_title('45 deg around Z')
    rot_z45 = R.from_euler('z', 45, degrees=True).as_matrix()
    plot_frame(ax2, [0, 0, 0], np.eye(3), alpha=0.3)
    plot_frame(ax2, [0, 0, 0], rot_z45)
    ax2.set_xlim([-1, 1]); ax2.set_ylim([-1, 1]); ax2.set_zlim([-1, 1])
    
    # Multiple rotations
    ax3 = fig.add_subplot(133, projection='3d')
    ax3.set_title('Random Rotations')
    plot_frame(ax3, [0, 0, 0], np.eye(3), alpha=0.3)
    for _ in range(5):
        rot = R.random().as_matrix()
        plot_frame(ax3, [0, 0, 0], rot, alpha=0.6)
    ax3.set_xlim([-1, 1]); ax3.set_ylim([-1, 1]); ax3.set_zlim([-1, 1])
    
    plt.tight_layout()
    plt.show()

visualize_rotations()

## 4. Diffusion on SO(3)

For protein generation, we need to add noise to rotations. Standard Gaussian noise doesn't work because rotations live on a manifold.

We use the **Isotropic Gaussian on SO(3)** (IGSO3):
$$p(R | \sigma) \propto \exp\left(-\frac{\omega^2}{2\sigma^2}\right)$$

where $\omega$ is the rotation angle.

In [ ]:
def sample_igso3(n_samples, sigma):
    """
    Sample from Isotropic Gaussian on SO(3).
    
    Args:
        n_samples: number of samples
        sigma: concentration parameter
    
    Returns:
        rotations: [n_samples, 3, 3]
    """
    # Sample rotation angle from folded normal
    angles = np.abs(np.random.normal(0, sigma, n_samples))
    angles = np.clip(angles, 0, np.pi)  # Clip to valid range
    
    # Sample random axis uniformly on S^2
    axes = np.random.randn(n_samples, 3)
    axes = axes / np.linalg.norm(axes, axis=1, keepdims=True)
    
    # Create rotations
    rotations = [R.from_rotvec(angle * axis).as_matrix() 
                 for angle, axis in zip(angles, axes)]
    
    return np.stack(rotations)

# Visualize IGSO3 samples at different sigma values
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

sigmas = [0.1, 0.5, 1.0, 2.0]

for ax, sigma in zip(axes, sigmas):
    samples = sample_igso3(1000, sigma)
    
    # Compute rotation angles
    traces = np.trace(samples, axis1=1, axis2=2)
    angles = np.arccos(np.clip((traces - 1) / 2, -1, 1))
    
    ax.hist(np.degrees(angles), bins=30, density=True, alpha=0.7)
    ax.set_xlabel('Rotation Angle (degrees)')
    ax.set_ylabel('Density')
    ax.set_title(f'IGSO3, sigma={sigma}')
    ax.set_xlim([0, 180])

plt.tight_layout()
plt.show()

In [ ]:
def visualize_so3_diffusion():
    """Visualize the forward diffusion process on SO(3)."""
    fig = plt.figure(figsize=(15, 5))
    
    # Start with identity
    R0 = np.eye(3)
    
    timesteps = [0.0, 0.25, 0.5, 0.75, 1.0]
    
    for idx, t in enumerate(timesteps):
        ax = fig.add_subplot(1, 5, idx+1, projection='3d')
        ax.set_title(f't = {t}')
        
        # Plot original frame (faded)
        plot_frame(ax, [0, 0, 0], R0, alpha=0.2)
        
        # Sample noisy rotations
        sigma = t * 1.5  # Linear schedule
        if sigma > 0:
            for _ in range(10):
                noise_rot = sample_igso3(1, sigma)[0]
                noisy_R = noise_rot @ R0
                plot_frame(ax, [0, 0, 0], noisy_R, alpha=0.4)
        else:
            plot_frame(ax, [0, 0, 0], R0)
        
        ax.set_xlim([-1.2, 1.2])
        ax.set_ylim([-1.2, 1.2])
        ax.set_zlim([-1.2, 1.2])
    
    plt.suptitle('Forward Diffusion on SO(3)', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_so3_diffusion()

## 5. SE(3) Invariant Features

For equivariant networks, we need features that don't change under SE(3) transformations.

**Invariant quantities:**
- Distances: $\|\vec{x}_i - \vec{x}_j\|$
- Angles between vectors
- Relative orientations (trace of $R_i^T R_j$)

In [ ]:
def compute_invariant_features(frames_i, frames_j):
    """
    Compute SE(3)-invariant features between pairs of frames.
    
    Args:
        frames_i, frames_j: RigidTransform objects
    
    Returns:
        dict of invariant features
    """
    features = {}
    
    # Distance (invariant)
    diff = frames_i.trans - frames_j.trans
    distance = diff.norm(dim=-1)
    features['distance'] = distance
    
    # Relative rotation: R_rel = R_i^T @ R_j
    R_rel = torch.einsum('...ij,...jk->...ik', 
                         frames_i.rot.transpose(-1, -2),
                         frames_j.rot)
    
    # Trace of relative rotation (invariant, related to rotation angle)
    trace = R_rel[..., 0, 0] + R_rel[..., 1, 1] + R_rel[..., 2, 2]
    features['rel_rot_trace'] = trace
    
    # Rotation angle
    rot_angle = torch.acos(torch.clamp((trace - 1) / 2, -1 + 1e-7, 1 - 1e-7))
    features['rel_rot_angle'] = rot_angle
    
    return features

# Create two frames
R1 = axis_angle_to_rotation_matrix(torch.tensor([0.0, 0.0, 0.5]))
R2 = axis_angle_to_rotation_matrix(torch.tensor([0.3, 0.0, 0.0]))

frame1 = RigidTransform(R1, torch.tensor([0.0, 0.0, 0.0]))
frame2 = RigidTransform(R2, torch.tensor([3.0, 4.0, 0.0]))

# Compute invariant features
features = compute_invariant_features(frame1, frame2)
print("Original features:")
for k, v in features.items():
    print(f"  {k}: {v.item():.4f}")

# Apply a random SE(3) transformation to both frames
random_rot = axis_angle_to_rotation_matrix(torch.randn(3))
random_trans = torch.randn(3) * 10
T_random = RigidTransform(random_rot, random_trans)

frame1_transformed = T_random.compose(frame1)
frame2_transformed = T_random.compose(frame2)

# Compute features after transformation
features_after = compute_invariant_features(frame1_transformed, frame2_transformed)
print("\nFeatures after SE(3) transformation (should be same):")
for k, v in features_after.items():
    print(f"  {k}: {v.item():.4f}")

## 6. SE(3) Equivariant Layers

An equivariant layer satisfies:
$$f(T \cdot x) = T \cdot f(x)$$

We achieve this by:
1. Computing messages using invariant features
2. Transforming outputs in local coordinate frames

In [ ]:
class SE3EquivariantLayer(nn.Module):
    """Simple SE(3) equivariant message passing layer."""
    
    def __init__(self, node_dim, hidden_dim=64):
        super().__init__()
        
        # Invariant feature dimension: distance (1) + angle features (3)
        inv_dim = 4
        
        # Message network (operates on invariant features)
        self.message_net = nn.Sequential(
            nn.Linear(node_dim * 2 + inv_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim)
        )
        
        # Position update (predicts displacement in local frame)
        self.pos_net = nn.Sequential(
            nn.Linear(node_dim * 2 + inv_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3)
        )
        
    def forward(self, node_features, frames, edge_index):
        """
        Args:
            node_features: [N, node_dim]
            frames: RigidTransform with [N] frames
            edge_index: [2, E] edges
        
        Returns:
            new_node_features: [N, node_dim]
            pos_updates: [N, 3]
        """
        src, dst = edge_index
        N = node_features.shape[0]
        
        # Compute invariant edge features
        frame_src = RigidTransform(frames.rot[src], frames.trans[src])
        frame_dst = RigidTransform(frames.rot[dst], frames.trans[dst])
        
        inv_features = compute_invariant_features(frame_src, frame_dst)
        inv_feat_tensor = torch.stack([
            inv_features['distance'],
            inv_features['rel_rot_trace'],
            torch.sin(inv_features['rel_rot_angle']),
            torch.cos(inv_features['rel_rot_angle'])
        ], dim=-1)
        
        # Compute messages
        msg_input = torch.cat([
            node_features[src],
            node_features[dst],
            inv_feat_tensor
        ], dim=-1)
        
        messages = self.message_net(msg_input)
        
        # Aggregate messages
        aggregated = torch.zeros_like(node_features)
        aggregated.scatter_add_(0, dst.unsqueeze(-1).expand(-1, node_features.shape[-1]), messages)
        
        # Compute position updates in local frame
        pos_updates_local = self.pos_net(msg_input)
        
        # Transform to global frame
        pos_updates_global = frame_src.apply(pos_updates_local) - frame_src.trans
        
        # Aggregate position updates
        pos_aggregated = torch.zeros(N, 3)
        pos_aggregated.scatter_add_(0, dst.unsqueeze(-1).expand(-1, 3), pos_updates_global)
        
        return node_features + aggregated, pos_aggregated

# Test the layer
N = 10  # Number of nodes
node_dim = 32

node_features = torch.randn(N, node_dim)
rots = torch.stack([axis_angle_to_rotation_matrix(torch.randn(3) * 0.5) for _ in range(N)])
trans = torch.randn(N, 3) * 5
frames = RigidTransform(rots, trans)

# Create edges (fully connected for simplicity)
edge_index = torch.stack([
    torch.repeat_interleave(torch.arange(N), N),
    torch.arange(N).repeat(N)
])
# Remove self-loops
mask = edge_index[0] != edge_index[1]
edge_index = edge_index[:, mask]

layer = SE3EquivariantLayer(node_dim)
new_features, pos_updates = layer(node_features, frames, edge_index)

print(f"Input node features: {node_features.shape}")
print(f"Output node features: {new_features.shape}")
print(f"Position updates: {pos_updates.shape}")

In [ ]:
# Test equivariance
def test_equivariance(layer, node_features, frames, edge_index):
    """Test if layer is SE(3) equivariant."""
    # Random SE(3) transformation
    T_rand = RigidTransform(
        axis_angle_to_rotation_matrix(torch.randn(3)),
        torch.randn(3) * 5
    )
    
    # Forward on original
    out_features, out_pos = layer(node_features, frames, edge_index)
    
    # Transform outputs (node features are invariant, positions are equivariant)
    out_pos_transformed = T_rand.apply(out_pos)
    
    # Transform inputs
    frames_transformed = T_rand.compose(frames)
    
    # Forward on transformed
    out_features2, out_pos2 = layer(node_features, frames_transformed, edge_index)
    
    # Check equivariance
    feat_error = (out_features - out_features2).abs().max().item()
    pos_error = (out_pos_transformed - out_pos2).abs().max().item()
    
    print(f"Feature difference (should be ~0): {feat_error:.6f}")
    print(f"Position difference (should be ~0): {pos_error:.6f}")
    
    return feat_error < 1e-5 and pos_error < 1e-5

# Run test
with torch.no_grad():
    is_equivariant = test_equivariance(layer, node_features, frames, edge_index)
    print(f"\nLayer is SE(3) equivariant: {is_equivariant}")

## 7. Protein Backbone as Frames

In proteins, each residue can be represented as a rigid frame defined by the N-CA-C atoms.

In [ ]:
def frames_from_backbone(N_coords, CA_coords, C_coords):
    """
    Compute rigid frames from backbone coordinates.
    
    The frame is defined as:
    - Origin: CA position
    - X-axis: CA -> C direction
    - Y-axis: perpendicular to X in the N-CA-C plane
    - Z-axis: cross product of X and Y
    
    Args:
        N_coords: [L, 3] N atom coordinates
        CA_coords: [L, 3] CA atom coordinates
        C_coords: [L, 3] C atom coordinates
    
    Returns:
        frames: RigidTransform with [L] frames
    """
    # Translation: CA position
    trans = CA_coords
    
    # X-axis: CA -> C
    x_axis = C_coords - CA_coords
    x_axis = x_axis / (x_axis.norm(dim=-1, keepdim=True) + 1e-8)
    
    # Vector in plane: CA -> N
    v = N_coords - CA_coords
    v = v / (v.norm(dim=-1, keepdim=True) + 1e-8)
    
    # Z-axis: perpendicular to plane
    z_axis = torch.cross(x_axis, v, dim=-1)
    z_axis = z_axis / (z_axis.norm(dim=-1, keepdim=True) + 1e-8)
    
    # Y-axis: complete right-handed frame
    y_axis = torch.cross(z_axis, x_axis, dim=-1)
    
    # Rotation matrix: columns are axes
    rot = torch.stack([x_axis, y_axis, z_axis], dim=-1)
    
    return RigidTransform(rot, trans)

# Create a simple alpha helix-like structure
L = 10  # Number of residues

# Ideal alpha helix parameters
rise_per_residue = 1.5  # Angstroms
radius = 2.3  # Angstroms
residues_per_turn = 3.6

# Generate backbone coordinates
t = torch.arange(L).float()
theta = 2 * np.pi * t / residues_per_turn

CA_coords = torch.stack([
    radius * torch.cos(theta),
    radius * torch.sin(theta),
    rise_per_residue * t
], dim=-1)

# N and C atoms are offset from CA
N_coords = CA_coords + torch.tensor([0.5, 0.0, -0.5])
C_coords = CA_coords + torch.tensor([-0.5, 0.0, 0.5])

# Compute frames
helix_frames = frames_from_backbone(N_coords, CA_coords, C_coords)

print(f"Number of frames: {helix_frames.rot.shape[0]}")
print(f"Frame rotation shape: {helix_frames.rot.shape}")
print(f"Frame translation shape: {helix_frames.trans.shape}")

In [ ]:
# Visualize the helix frames
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot CA trace
ax.plot(CA_coords[:, 0], CA_coords[:, 1], CA_coords[:, 2], 
        'k-', linewidth=2, label='Backbone')
ax.scatter(CA_coords[:, 0], CA_coords[:, 1], CA_coords[:, 2], 
           s=50, c='black', label='CA atoms')

# Plot frames every 2 residues
for i in range(0, L, 2):
    origin = helix_frames.trans[i].numpy()
    rot = helix_frames.rot[i].numpy()
    plot_frame(ax, origin, rot, scale=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Alpha Helix with Residue Frames')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Diffusion on Protein Frames

Let's implement the forward diffusion process for protein frames.

In [ ]:
def diffuse_frames(frames, t, sigma_trans=10.0, sigma_rot=1.5):
    """
    Add noise to SE(3) frames.
    
    Args:
        frames: RigidTransform
        t: timestep in [0, 1]
        sigma_trans: max translation noise
        sigma_rot: max rotation noise
    
    Returns:
        noisy_frames: RigidTransform
    """
    L = frames.rot.shape[0]
    
    # Translation noise (Gaussian)
    trans_noise = torch.randn(L, 3) * sigma_trans * t
    noisy_trans = frames.trans + trans_noise
    
    # Rotation noise (IGSO3)
    noise_rots = sample_igso3(L, sigma_rot * t)
    noise_rots = torch.tensor(noise_rots, dtype=torch.float32)
    noisy_rots = torch.einsum('...ij,...jk->...ik', noise_rots, frames.rot)
    
    return RigidTransform(noisy_rots, noisy_trans)

# Visualize diffusion process
fig, axes = plt.subplots(1, 5, figsize=(20, 4), subplot_kw={'projection': '3d'})

timesteps = [0.0, 0.25, 0.5, 0.75, 1.0]

for ax, t in zip(axes, timesteps):
    if t == 0:
        noisy_frames = helix_frames
    else:
        noisy_frames = diffuse_frames(helix_frames, t)
    
    # Plot backbone
    coords = noisy_frames.trans.numpy()
    ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], 'k-', linewidth=1)
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=30, c='blue')
    
    # Plot frames
    for i in range(0, L, 2):
        origin = noisy_frames.trans[i].numpy()
        rot = noisy_frames.rot[i].numpy()
        plot_frame(ax, origin, rot, scale=0.3)
    
    ax.set_title(f't = {t}')
    ax.set_xlim([-15, 15])
    ax.set_ylim([-15, 15])
    ax.set_zlim([-5, 20])

plt.suptitle('Forward Diffusion on Protein Frames', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Key Takeaways

1. **SE(3) Group**: Combines rotations (SO(3)) and translations (R^3) - the natural symmetry group for 3D structures.

2. **Rotation Representations**: Quaternions are numerically stable; axis-angle is intuitive; rotation matrices are easy to apply.

3. **Invariant Features**: Distances and relative orientations don't change under SE(3) transformations.

4. **Equivariant Networks**: Use invariant features for computation, transform outputs in local frames.

5. **SO(3) Diffusion**: Use IGSO3 distribution instead of Gaussian noise for rotation diffusion.

6. **Frame Representation**: Proteins are naturally represented as rigid frames at each residue.

## 10. Exercises

1. **Implement SLERP**: Implement spherical linear interpolation for rotations to smoothly interpolate between two orientations.

2. **Add more invariant features**: Extend the edge features to include dihedral angles and other geometric quantities.

3. **Test with real proteins**: Load a PDB file and compute frames from real backbone coordinates.

4. **Implement denoising**: Create a simple denoising network that predicts the clean structure from noisy frames.

In [ ]:
# Exercise 1: Implement SLERP

def slerp(q1, q2, t):
    """
    Spherical linear interpolation between quaternions.
    
    Args:
        q1: [..., 4] starting quaternion
        q2: [..., 4] ending quaternion
        t: interpolation parameter (0 = q1, 1 = q2)
    
    Returns:
        q: [..., 4] interpolated quaternion
    """
    # Normalize
    q1 = q1 / (q1.norm(dim=-1, keepdim=True) + 1e-8)
    q2 = q2 / (q2.norm(dim=-1, keepdim=True) + 1e-8)
    
    # Compute cosine of angle
    dot = (q1 * q2).sum(dim=-1, keepdim=True)
    
    # If negative dot, negate one quaternion (shorter path)
    q2 = torch.where(dot < 0, -q2, q2)
    dot = torch.abs(dot)
    
    # Clamp for numerical stability
    dot = torch.clamp(dot, -1 + 1e-7, 1 - 1e-7)
    
    # Compute angle
    theta = torch.acos(dot)
    sin_theta = torch.sin(theta)
    
    # Interpolate
    s1 = torch.sin((1 - t) * theta) / (sin_theta + 1e-8)
    s2 = torch.sin(t * theta) / (sin_theta + 1e-8)
    
    q = s1 * q1 + s2 * q2
    return q / (q.norm(dim=-1, keepdim=True) + 1e-8)

# Test SLERP
q1 = torch.tensor([1.0, 0.0, 0.0, 0.0])  # Identity
q2 = torch.tensor([np.cos(np.pi/4), 0.0, 0.0, np.sin(np.pi/4)])  # 90 deg around z

print("SLERP interpolation:")
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    q = slerp(q1, q2, t)
    R = quaternion_to_rotation_matrix(q)
    angle = np.degrees(np.arccos((R.trace() - 1) / 2))
    print(f"  t={t:.2f}: rotation angle = {angle:.1f} degrees")